Step 1: Dataset Acquisition & Loading: 
This step downloads the public-domain Tiny Shakespeare text dataset and loads it into memory so we can inspect its contents.

In [1]:
import urllib.request

# Download a small public-domain dataset (Tiny Shakespeare sample)
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
dataset_path = "dataset.txt"
urllib.request.urlretrieve(url, dataset_path)

# Load and inspect the dataset
with open(dataset_path, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Dataset loaded successfully!")
print(f"Total characters in dataset: {len(text):,}")
print("\n--- First 300 characters ---")
print(text[:300])

Dataset loaded successfully!
Total characters in dataset: 1,115,394

--- First 300 characters ---
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


Vocabulary Creation & Text Encoding: 
Here, we extract all unique characters to build our vocabulary, create encoding/decoding mappings, and convert the entire raw text string into a PyTorch tensor of integers.

In [2]:
import torch

# Build character vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocabulary size (unique characters): {vocab_size}")

# Create mapping from characters to integers and vice versa
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s] # encoder: string -> list of ints
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: list of ints -> string

# Encode the entire text dataset and store it into a PyTorch tensor
data = torch.tensor(encode(text), dtype=torch.long)
print(f"Data tensor shape: {data.shape}")

Vocabulary size (unique characters): 65
Data tensor shape: torch.Size([1115394])


Train/Validation Split & Batching Pipeline: 
We split our dataset (90% for training, 10% for validation) and set up a batching function to feed small chunks of text into our model during training.

In [3]:
# Hyperparameters
batch_size = 32
block_size = 8
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(1337)

# Create separate training and validation data
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    # Generate a small batch of data of inputs x and targets y
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

print(f"Using device: {device}")
xb, yb = get_batch('train')
print("Inputs x shape:", xb.shape)
print("Targets y shape:", yb.shape)

Using device: cpu
Inputs x shape: torch.Size([32, 8])
Targets y shape: torch.Size([32, 8])


Model Architecture Definition: 
This defines our character-level Bigram Language Model using PyTorch embedding tables and a text generation function to sample new characters iteratively.

In [4]:
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Each token reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) # (Batch, Time, Vocab_size)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :] # Focus on the last time step
            probs = F.softmax(logits, dim=-1) # Get probabilities
            idx_next = torch.multinomial(probs, num_samples=1) # Sample next token
            idx = torch.cat((idx, idx_next), dim=1) # Append to context
        return idx

model = BigramLanguageModel(vocab_size).to(device)

Training Loop & Loss Tracking: 
We train our model using the AdamW optimizer, periodically calculating and recording the training and validation loss results to monitor convergence.

In [5]:
max_iters = 3000
eval_interval = 300
learning_rate = 1e-2
eval_iters = 200

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

print("--- TRAINING LOSS RESULTS ---")
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

--- TRAINING LOSS RESULTS ---
step 0: train loss 4.7676, val loss 4.7677
step 300: train loss 2.8381, val loss 2.8538
step 600: train loss 2.5531, val loss 2.5811
step 900: train loss 2.4967, val loss 2.5151
step 1200: train loss 2.4878, val loss 2.5086
step 1500: train loss 2.4677, val loss 2.4946
step 1800: train loss 2.4695, val loss 2.4961
step 2100: train loss 2.4707, val loss 2.4871
step 2400: train loss 2.4645, val loss 2.4892
step 2700: train loss 2.4736, val loss 2.4920
step 2999: train loss 2.4617, val loss 2.4907


Model Serialization (Saving the Model): 
Once training is finished, we save the trained weights of our model to a file (character_level_bigram_model.pth) as required.

In [6]:
model_filename = "character_level_bigram_model.pth"
torch.save(model.state_dict(), model_filename)
print(f"\nModel successfully saved as '{model_filename}'")


Model successfully saved as 'character_level_bigram_model.pth'


Generating Ten Samples from a Starter Sequence: 
Finally, we use our trained model to generate and output ten distinct text patterns starting from a given sequence token.

In [7]:
print("\n--- TEN GENERATED SAMPLES ---")
starter_text = "\n"  # Starting character sequence
context_encoded = torch.tensor([encode(starter_text)], dtype=torch.long, device=device)

for i in range(10):
    generated_idx = model.generate(context_encoded, max_new_tokens=100)
    sample_text = decode(generated_idx[0].tolist())
    print(f"\nSample {i+1}:\n{sample_text}")


--- TEN GENERATED SAMPLES ---

Sample 1:

THAr toud en:moucr come

Pat s, ore wilkigulllo isowanornon ws alinthifofr
Cuker:
Thinertor wien IUC

Sample 2:

Fokil t.
And! he s thaswsantld was w,
Aue y s Ind benteecofo thime me,


API Win he qur s gtht t cam

Sample 3:

Waneld ol ce.
PUCite
Harenthrvosen ndizGouncesere ballerrang; Vator o he t mare.
Tuc;
Houd wnad; fou

Sample 4:

AURWhend, we lyoundre k beZARo g ls we MII:
Powor mese's we ant westh afre CI ond.
Dqup fof ut wet d

Sample 5:

I: s t a ?
NI s,
TCHe
So me p:
swibes sofurocatt.


Toubafr
Anous gar, byor.
O te,
Sis ldou bicorthi

Sample 6:

I f!
Gosthare! cthoth,
An bilA:
Ayalfo I
cent o we, bret us
AUETIs; hether thext tho'd dichtat tor t

Sample 7:


IURORI ssowin Whe, t be t we, mat yse in ndur.
FRora?
IED rayovis or;
Touthery thed move athe y how

Sample 8:

LUENal acoNETh! ms
LI'd che, gat f m.
NIV:
OLILAs he l,

Baly athigaloruguno fey brcer'lend maino wi

Sample 9:

Whecay!
Frciadsour nofrdmpo-lp' moue cer ds,
Thisord
